In [1]:
import json
import struct
import numpy as np
import matplotlib.pyplot as plt
import photonforge as pf
import siepic_forge as siepic
import luxtelligence_lnoi400_forge as lxt
import tidy3d as td

td.config.logging.level = "ERROR"

# Set up technologies
siepic_tech = siepic.ebeam()
basic_tech = pf.basic_technology()
lxt_tech = lxt.lnoi400()
pf.config.default_technology = siepic_tech

# Initialize live viewer for real-time visualization
from photonforge.live_viewer import LiveViewer
viewer = LiveViewer()

# Define simulation parameters
wavelengths = np.linspace(1.53, 1.57, 101)
freqs = pf.C_0 / wavelengths

21:26:29 SE Asia Standard Time WARNING: Using canonical configuration directory 
                               at 'C:\Users\James\.config\tidy3d'. Found legacy 
                               directory at 'C:\Users\James\.tidy3d', which will
                               be ignored. Tidy3D configuration now uses        
                               'C:\Users\James\.config\tidy3d\config.toml'.     

21:26:35 SE Asia Standard Time WARNING: The material-library variant            
                               'Palik_Lossless' is deprecated and maps to       
                               'Palik_LowLoss' because it contains a tiny fitted
                               loss despite its name. Use 'Palik_NoLoss' where  
                               available for a zero-loss Palik model.           

LiveViewer started at http://localhost:55402


In [2]:
dual_mode_spec = siepic_tech.ports["TE_1550_500"].copy()
dual_mode_spec.num_modes = 2  # Use both modes

siepic_tech.add_port("TE-TM_1550_500", dual_mode_spec)
siepic_tech.ports["TE-TM_1550_500"]

PortSpec(description="Strip TE 1550 nm, w=500 nm", width=1.5, limits=(-0.6, 0.82), num_modes=2, added_solver_modes=0, polarization="", target_neff=3.5, default_radius=0, path_profiles=[(0.5, 0, (1, 0))])

In [3]:
plasmonic_gold_gap = 0.03  # Gap between waveguide and gold
plasmonic_gold_thickness = 0.14  # Thickness of the gold layer
wg_height = 0.22

plasmonic_gold_layer = pf.LayerSpec(layer=(15, 0), description="Metal", color="FFE747", pattern="xx")

siepic_tech.add_layer("Plasmonic Gold Top", plasmonic_gold_layer)

plasmonic_gold_extrusion = pf.ExtrusionSpec(mask_spec=pf.MaskSpec((15, 0)), 
                                            medium={"optical": td.material_library['Au']['JohnsonChristy1972'], 
                                                    "electrical": td.LossyMetalMedium(conductivity=17.0, fit_param={'attrs': {}, 'max_num_poles': 16, 'tolerance_rms': 0.001, 'frequency_sampling_points': 20, 'log_sampling': True, 'type': 'SurfaceImpedanceFitterParam'}, frequency_range=(100000000.0, 200000000000.0))}, 
                                                    limits=(wg_height+plasmonic_gold_gap, wg_height+plasmonic_gold_gap+plasmonic_gold_thickness), sidewall_angle=0, reference="top")
siepic_tech.insert_extrusion_spec(4, plasmonic_gold_extrusion)

Name,Layer,Description,Color,Pattern
Si,"(1, 0)",SiEPIC - Waveguide,#ff80a818,\\
PinRec,"(1, 10)",SiEPIC,#ff80a818,xx
PinRecM,"(1, 11)",SiEPIC,#80000018,+
Si Slab,"(2, 0)",Dedicated Run Layers - Device…… Layer Partial Etch,#c080ff18,/
Direct Metal,"(5, 0)",Dedicated Run Layers,#80a8ff18,||
Oxide open to BOX,"(6, 0)",Dedicated Run Layers,#ff000018,-
Text,"(10, 0)",Text-Not Fabricated,#00000018,hollow
M1_heater,"(11, 0)",TiW Heater,#0000ff18,\\
M2_router,"(12, 0)",TiW/Au Routing Bilayer,#ffbf0018,//
M_Open,"(13, 0)",Bond Pad Open,#80005718,\\


In [4]:
plasmonic_gold_gap = 0.03  # Gap between waveguide and gold
plasmonic_gold_thickness = 0.14  # Thickness of the gold layer
wg_height = 0.22

plasmonic_gold_layer = pf.LayerSpec(layer=(16, 0), description="Metal", color="FFE747", pattern="xx")

siepic_tech.add_layer("Plasmonic Gold Side", plasmonic_gold_layer)

plasmonic_gold_extrusion = pf.ExtrusionSpec(mask_spec=pf.MaskSpec((16, 0)), 
                                            medium={"optical": td.material_library['Au']['JohnsonChristy1972'], 
                                                    "electrical": td.LossyMetalMedium(conductivity=17.0, fit_param={'attrs': {}, 'max_num_poles': 16, 'tolerance_rms': 0.001, 'frequency_sampling_points': 20, 'log_sampling': True, 'type': 'SurfaceImpedanceFitterParam'}, frequency_range=(100000000.0, 200000000000.0))}, 
                                                    limits=(wg_height/2, wg_height), sidewall_angle=0, reference="top")
siepic_tech.insert_extrusion_spec(5, plasmonic_gold_extrusion)

Name,Layer,Description,Color,Pattern
Si,"(1, 0)",SiEPIC - Waveguide,#ff80a818,\\
PinRec,"(1, 10)",SiEPIC,#ff80a818,xx
PinRecM,"(1, 11)",SiEPIC,#80000018,+
Si Slab,"(2, 0)",Dedicated Run Layers - Device…… Layer Partial Etch,#c080ff18,/
Direct Metal,"(5, 0)",Dedicated Run Layers,#80a8ff18,||
Oxide open to BOX,"(6, 0)",Dedicated Run Layers,#ff000018,-
Text,"(10, 0)",Text-Not Fabricated,#00000018,hollow
M1_heater,"(11, 0)",TiW Heater,#0000ff18,\\
M2_router,"(12, 0)",TiW/Au Routing Bilayer,#ffbf0018,//
M_Open,"(13, 0)",Bond Pad Open,#80005718,\\


In [5]:
class ThermalModel(pf.Model):
    def __init__(self, n_complex, voltage=0, coefficient=3e-4):
        super().__init__(
            n_complex=n_complex,
            voltage=voltage,
            coefficient=coefficient,
        )
        self.n_complex = np.array(n_complex, ndmin=2)
        self.voltage = voltage
        self.coefficient = coefficient

    def __copy__(self):
        return ThermalModel(self.n_complex, self.voltage, self.coefficient)

    def __deepcopy__(self, memo=None):
        # n_complex is an array, so we want to make sure to create a deep copy of it.
        # Other values (voltage and coefficient) are immutable (floats), so we can use them directly.
        return ThermalModel(self.n_complex.copy(), self.voltage, self.coefficient)

    def __repr__(self):
        return f"ThermalModel({self.n_complex!r}, {self.voltage!r}, {self.coefficient!r})"

    def __str__(self):
        return f"ThermalModel at {self.voltage} V"

    @property
    def as_bytes(self):
        coeffs = struct.pack("<2d", self.voltage, self.coefficient)
        shape = struct.pack("<2l", *self.n_complex.shape)
        n_data = self.n_complex.astype(complex).tobytes()
        # Add version 0 as first byte
        return b"\x00" + coeffs + shape + n_data

    @classmethod
    def from_bytes(cls, byte_repr):
        version = byte_repr[0]
        if version != 0:
            raise RuntimeError(f"Incompatible version for ThermalModel: {version}")

        byte_repr = byte_repr[1:]
        fmt = "<2d2l"
        head_len = struct.calcsize(fmt)
        voltage, coefficient, rows, cols = struct.unpack(fmt, byte_repr[:head_len])

        byte_repr = byte_repr[head_len:]
        n_complex = np.frombuffer(byte_repr, dtype=complex).reshape((rows, cols))

        return cls(n_complex, voltage, coefficient)

    @pf.cache_s_matrix
    def start(self, component, frequencies, voltage=None, **kwargs):
        # Allow overriding voltage as an `s_matrix` kwarg too
        if voltage is None:
            voltage = self.voltage
        n_complex = self.n_complex + self.coefficient * voltage**2
        wg_model = pf.WaveguideModel(n_complex)
        return wg_model.start(component, frequencies, **kwargs)


pf.register_model_class(ThermalModel)

In [6]:
@pf.parametric_component
def create_phase_shifter(port_spec="TE_1550_500", length=100):
    component = pf.Component("ps")

    phase_shifter = pf.parametric.straight(name="ps", port_spec=port_spec, length=length)
    str_ref = component.add_reference(phase_shifter)

    # Solve for the port mode of the waveguide and extract the complex refractive index
    alpha = 10
    kappa = (alpha * wavelengths * 1e-4 * np.log(10)) / (40 * np.pi)
    mode_solver = pf.port_modes(port=phase_shifter.ports["P0"], frequencies=freqs)
    n_complex = mode_solver.data.n_complex.values.T + 1j * kappa  # add propagation loss

    thermal_model = ThermalModel(n_complex=n_complex)
    component.add_model(thermal_model, "Thermal")

    component.add_port(component.detect_ports([port_spec]))
    component.add_model(pf.CircuitModel(), "CircuitModel")

    # heaters
    terminal_width = 10
    heater_width = 2

    heater = (
        pf.Path((str_ref.x_min, str_ref.y_mid), heater_width)
        .segment((str_ref.x_max, str_ref.y_mid), heater_width)
    )

    route_vp = (
        pf.Path((str_ref.x_min-15-terminal_width/2, str_ref.y_mid), terminal_width)
        .segment((str_ref.x_min-15+terminal_width/2, str_ref.y_mid), terminal_width)
        .segment((str_ref.x_min, str_ref.y_mid), heater_width)
    )

    route_vn = (
        pf.Path((str_ref.x_max, str_ref.y_mid), heater_width)
        .segment((str_ref.x_max+15-terminal_width/2, str_ref.y_mid), terminal_width)
        .segment((str_ref.x_max+15+terminal_width/2, str_ref.y_mid), terminal_width)
    )

    component.add((11,0), heater)
    component.add((12,0), route_vp)
    component.add((12,0), route_vn)
    component.add_terminal(pf.Terminal((12,0), pf.Rectangle(size=(terminal_width, terminal_width), center=(str_ref.x_min-15, str_ref.y_mid))), "VP")
    component.add_terminal(pf.Terminal((12,0), pf.Rectangle(size=(terminal_width, terminal_width), center=(str_ref.x_max+15, str_ref.y_mid))), "VN")

    return component

ps = create_phase_shifter()
viewer(ps)

                               Loading simulation from local cache. View cached 
                               task using web UI at                             
                               ]8;id=982381;https://tidy3d.simulation.cloud/workbench?taskId=mo-0abc5c2e-0424-4551-8fb0-966e2e82dbe4\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=944552;https://tidy3d.simulation.cloud/workbench?taskId=mo-0abc5c2e-0424-4551-8fb0-966e2e82dbe4\taskId]8;;\
                               ]8;id=982381;https://tidy3d.simulation.cloud/workbench?taskId=mo-0abc5c2e-0424-4551-8fb0-966e2e82dbe4\=]8;;\]8;id=567426;https://tidy3d.simulation.cloud/workbench?taskId=mo-0abc5c2e-0424-4551-8fb0-966e2e82dbe4\mo]8;;\]8;id=982381;https://tidy3d.simulation.cloud/workbench?taskId=mo-0abc5c2e-0424-4551-8fb0-966e2e82dbe4\-0abc5c2e-0424-4551-8fb0-966e2e82dbe4']8;;\.

Progress: 100% 
